# Reference · Day 9 studio — a probability, a rule, and who you point them at

**Not a marking key.** You each used your own seed, so your numbers differ from these a
little; this notebook uses the class's split (seed 764) so it matches Tuesday's slides. If
your *pattern* differs from the one below, the interesting question is which choice you made
differently.

The studio asked three things: pick a threshold **and say why**; check whether precision
follows the base rate the way Bayes' rule says it should; and ask which unit — any
encounter, or a patient's first — a hospital actually predicts for.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import numpy as np
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Meeting 9's problem: readmitted within 30 days, death and hospice discharges removed.
ENDED = [11, 13, 14, 19, 20, 21]
raw = diabetes()
d = raw[~raw["discharge_disposition_id"].isin(ENDED)].copy()
d["readmit30"] = (d["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
d[CAT] = d[CAT].astype(str)


def pipeline():
    """The pipeline since Meeting 2 -- fill gaps, scale, one-hot encode, then the model."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000)),
    ])


seed = 764        # the class's split. ⚠ Whatever seed you set, it has to REACH the split below.

## Question 1 · your baseline, your threshold

In [ ]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
              .split(d, groups=d["patient_nbr"]))          # every patient on one side
train, test = d.iloc[tr], d.iloc[te]
model = pipeline().fit(train[NUM + CAT], train["readmit30"])
p = model.predict_proba(test[NUM + CAT])[:, 1]             # column 1: P(readmitted)
y = test["readmit30"].to_numpy()

MY_THRESHOLD = 0.2
print(f"  seed {seed}: {len(test):,} test encounters, prevalence {y.mean():.3f}, "
      f"AUC {roc_auc_score(y, p):.3f}\n")
for t in (0.5, MY_THRESHOLD):
    tn, fp, fn, tp = confusion_matrix(y, p >= t).ravel()
    print(f"  threshold {t}:  TP {tp:>5}  FP {fp:>5}  FN {fn:>5}  TN {tn:>6}   "
          f"flags {(tp + fp) / len(y):.1%}   sensitivity {tp / (tp + fn):.3f}   "
          f"precision {tp / (tp + fp):.3f}")

### What this shows

At 0.5 the rule flags 0.4% of encounters and catches 48 of 2,739 readmissions. At 0.2 it
flags about one encounter in twelve and a quarter of them come back. **Why 0.2?** A list a
discharge team could plausibly work through — which is a guess about capacity, not a fact
about the model. The class's thresholds ran from 0.10 to 0.30, and the reasons were all of
this kind: "a missed readmission costs more than a false alarm", "prioritize precision",
"about 11% come back". Every one is a guess about prices; Meeting 10 turns it into a number.

Two things to check in your own notebook. If your seed was not 764 but your numbers match
these exactly, your seed never reached the split. And an AUC under one half — 0.345 here —
means `predict_proba(X)[:, 0]`, the probability of *not* coming back.

## Question 2 · does precision follow the base rate?

In [ ]:
prior = test["number_inpatient"].to_numpy() >= 1
flag = p >= MY_THRESHOLD
sens_all, spec_all = flag[y == 1].mean(), (~flag)[y == 0].mean()   # the whole test set

print(f"  {'prior inpatient stays':<22}{'prevalence':>11}{'sensitivity':>13}{'specificity':>13}"
      f"{'precision':>11}{'Section 6 said':>16}")
for label, g in [("none", ~prior), ("1 or more", prior)]:
    yy, ff = y[g], flag[g]
    rate = yy.mean()
    # Section 6's prediction: Bayes' rule, IF the whole set's sensitivity and specificity held.
    said = sens_all * rate / (sens_all * rate + (1 - spec_all) * (1 - rate))
    print(f"  {label:<22}{rate:>11.3f}{ff[yy == 1].mean():>13.3f}{(~ff)[yy == 0].mean():>13.3f}"
          f"{yy[ff].mean():>11.3f}{said:>16.3f}")

### What this shows

The base rate nearly doubles, from 0.086 to 0.162. Bayes' rule said precision should rise from
about 0.21 to 0.35 — **if** sensitivity and specificity were the same in both groups. They are
not: the model uses `number_inpatient`, so it is far more sensitive in the group with earlier
stays (0.325 against 0.070) and less specific. Precision barely moves, 0.241 to 0.260. **The
arithmetic was right; the assumption failed.**

Eight of your notebooks — eight seeds, five thresholds — all showed it: the base rate about
doubled, precision rose a little or fell, and never as far as Section 6 said; sensitivity in
the 1+ group was 2 to 15 times higher. Over twenty splits of the class data, Section 6's
arithmetic says 0.226 against 0.394 and the model gives 0.268 against 0.291; high-risk
precision is higher on only 15 of 20.

## Question 3 · one row per patient

In [ ]:
first = diabetes(patient_level=True)                       # each patient's first encounter
first = first[~first["discharge_disposition_id"].isin(ENDED)].copy()
first["readmit30"] = (first["readmitted"] == "<30").astype(int)
first[CAT] = first[CAT].astype(str)

f_train, f_test = train_test_split(first, test_size=0.25, random_state=seed)   # one row each
fp_ = pipeline().fit(f_train[NUM + CAT], f_train["readmit30"]).predict_proba(f_test[NUM + CAT])[:, 1]
fy = f_test["readmit30"].to_numpy()
ff = fp_ >= MY_THRESHOLD

print(f"  {'':<28}{'rows':>9}{'prevalence':>12}{'AUC':>8}{'flagged':>9}{'precision':>11}")
print(f"  {'every encounter (Q1)':<28}{len(d):>9,}{y.mean():>12.3f}{roc_auc_score(y, p):>8.3f}"
      f"{flag.mean():>9.3f}{y[flag].mean():>11.3f}")
print(f"  {'first encounter per patient':<28}{len(first):>9,}{fy.mean():>12.3f}"
      f"{roc_auc_score(fy, fp_):>8.3f}{ff.mean():>9.3f}{fy[ff].mean():>11.3f}")

### What this shows

A patient's first visit is a rarer event — prevalence about 0.09 against 0.11 — and the
ranking holds up. (With one row per patient, a plain random split already keeps each patient
on one side.) Your first-encounter AUCs ran from 0.644 to 0.665.

**Which question is the hospital's?** At discharge a hospital does not know whether this is a
patient's first visit or their fifth — but it does know their history. "Any encounter" is the
question it faces; "first encounter" is the cleaner statistical unit. Say which one your number
answers.

## Four questions to ask yourself

1. Did the seed I set actually reach the split?
2. Is my AUC from column 1 of `predict_proba` — the probability of coming back?
3. Is my threshold's reason about the hospital — a cost, a capacity — or about a metric?
4. Which unit does my number answer: any encounter, or a patient's first?